# Module 5: Critic-Refiner — Production Deployment

Deploy the Critic-Refiner specialist to Amazon Bedrock AgentCore Runtime (A2A protocol).  
The specialist runs the Writer↔Critic `GraphBuilder` loop internally.  
`chain.py` calls it directly — no orchestrator runtime is deployed.

---

## Step 1: Install dependencies

In [ ]:
%pip install "strands-agents[a2a]>=1.52.0" bedrock-agentcore boto3 httpx

---

## Step 2: Set up execution roles

Creates the IAM execution roles for the AgentCore runtimes (idempotent — safe to re-run).

In [ ]:
import sys, os, boto3
sys.path.insert(0, "../../shared")
import deploy_utils as u

session = u.get_session()
account = u.get_account(session)
bucket  = u.code_bucket_name(account, u.REGION)
iam     = session.client("iam", region_name=u.REGION)

# Create runtime role (Bedrock, Logs, S3) — used by the critic_refiner specialist
runtime_role_arn = u.ensure_runtime_role(
    iam, f"workshop-agentcore-m5-runtime-role",
    account, u.REGION, bucket,
)
os.environ["AGENTCORE_RUNTIME_ROLE_ARN"] = runtime_role_arn
print(f"Runtime role: {runtime_role_arn}")

---

## Step 3: Deploy

Deploys the critic_refiner specialist runtime (~3-5 min).  
No orchestrator runtime is deployed — `chain.py` calls the specialist directly.

In [ ]:
!python deploy.py --name-prefix m5

In [ ]:
import os

with open(".env_arns", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line.startswith("export "):
            key, _, val = line[len("export "):].partition("=")
            os.environ[key.strip()] = val.strip()

CRITIC_REFINER_ARN = os.environ["CRITIC_REFINER_RUNTIME_ARN"]
print(f"Critic-Refiner ARN: {CRITIC_REFINER_ARN}")

---

## Step 4: Run the chain

The ARN is set as an env var. `chain.py` calls the critic_refiner specialist directly — the specialist runs the Writer↔Critic `GraphBuilder` loop internally and returns the approved memo.

In [ ]:
import sys
sys.path.insert(0, ".")   # chain.py and a2a_utils.py are in the same folder

from chain import run_chain

BRIEF = (
    "NovaCart Premium Tier: Options A ($19.99/mo invite-only), "
    "B ($14.99/mo 5% pilot), C ($12.99/mo full launch). "
    "Target: +15% CLV in 6 months. Budget: $2M."
)

print("Running Critic-Refiner...")
print("─" * 60)
print(run_chain(BRIEF))

In [ ]:
# Run the chain from the terminal with a custom brief:
print("python chain.py")
print('python chain.py "your brief here"')

---

## Step 5: Observability

After running `chain.py`, traces appear in **CloudWatch > X-Ray > Traces** or **Amazon Bedrock > AgentCore > Observability**.

What you see per invocation:
- One span for the critic_refiner specialist call
- Writer and Critic spans nested inside, showing each cycle
- Total duration = time for the full Writer↔Critic loop to converge

No extra configuration needed: `aws-opentelemetry-distro` is in `requirements.txt` and AgentCore instruments the specialist automatically.

---

## Step 6: Cleanup

Uncomment and run the cell below to delete all AWS resources created by this module.

In [ ]:
# Uncomment and run to delete all resources created by this module.

# !python cleanup.py --name-prefix m5

# Verify:
# import boto3, os
# REGION = os.environ.get("AWS_REGION", "us-east-1")
# remaining = boto3.client("bedrock-agentcore-control", region_name=REGION).list_agent_runtimes()
# print([rt["agentRuntimeName"] for rt in remaining.get("agentRuntimes", [])])